# Quelle — Training on Colab

**Experiment-agnostic** disconnect-resistant training with periodic sync to Google Drive.

**Flow:** Mount Drive → Clone repo → Read experiment config → Install deps → Restore artifacts → Setup → Train (syncing to Drive) → Final sync.

On reconnect: re-run all cells. The notebook detects checkpoints and resumes automatically.

---
**Before running:** set `EXPERIMENT` in the Configuration cell below.

In [ ]:
# ============================================================
# Configuration — edit this cell
# ============================================================

REPO_URL    = "https://github.com/leonorae/quelle.git"
BRANCH      = "main"

# Which experiment to run (directory name under experiments/)
EXPERIMENT = "VVVVVV"  # or: "variable-bitrate-reasoning", "crystal-lattice"

# Drive root for all quelle artifacts
DRIVE_ROOT = "/content/drive/MyDrive/quelle_artifacts"

# Local paths (fast Colab SSD — lost on disconnect, hence the sync)
REPO_DIR = "/content/quelle"

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Clone / update repo

In [ ]:
import os, subprocess, sys

def run(cmd, **kwargs):
    """Run a shell command, streaming output, raising on failure."""
    print(f"$ {' '.join(cmd) if isinstance(cmd, list) else cmd}")
    result = subprocess.run(cmd, **kwargs)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")

if not os.path.exists(REPO_DIR):
    run(["git", "clone", "--recurse-submodules", REPO_URL, REPO_DIR])
    run(["git", "-C", REPO_DIR, "checkout", BRANCH])
else:
    run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH])
    run(["git", "-C", REPO_DIR, "checkout", BRANCH])
    run(["git", "-C", REPO_DIR, "merge", "--ff-only", f"origin/{BRANCH}"])
    run(["git", "-C", REPO_DIR, "submodule", "update", "--init"])

print(f"Repo: {REPO_DIR}")

## 3. Load experiment config

In [ ]:
import yaml
from pathlib import Path

exp_dir = Path(REPO_DIR) / "experiments" / EXPERIMENT
config_path = exp_dir / "colab.yaml"
if not config_path.exists():
    raise FileNotFoundError(f"No colab.yaml found at {config_path}")

with open(config_path) as f:
    CFG = yaml.safe_load(f)

DEP_MODE = CFG.get("dep_mode", "pip")
SLUG = CFG["slug"]
ARTIFACT_SUBDIR = CFG.get("artifact_subdir", SLUG)
DRIVE_ARTIFACT_DIR = os.path.join(DRIVE_ROOT, ARTIFACT_SUBDIR)
SYNC_INTERVAL_MIN = CFG.get("sync_interval_min", 5)

# For nanochat experiments, artifact source is the nanochat base dir
# For standalone experiments, it's the outputs/ directory
if DEP_MODE == "nanochat":
    LOCAL_ARTIFACT_DIR = f"/content/{SLUG}_artifacts"
else:
    LOCAL_ARTIFACT_DIR = str(exp_dir / "outputs")

print(f"Experiment: {SLUG}")
print(f"Description: {CFG.get('description', '')}")
print(f"Dep mode: {DEP_MODE}")
print(f"Drive artifacts: {DRIVE_ARTIFACT_DIR}")
print(f"Local artifacts: {LOCAL_ARTIFACT_DIR}")

## 4. Install dependencies

In [ ]:
# Fix: Colab's PyTorch requires libcusparseLt.so.0 from the system CUDA libs
_cuda_lib = "/usr/local/cuda/lib64"
if os.path.exists(_cuda_lib) and _cuda_lib not in os.environ.get("LD_LIBRARY_PATH", ""):
    os.environ["LD_LIBRARY_PATH"] = _cuda_lib + ":" + os.environ.get("LD_LIBRARY_PATH", "")
    print(f"Added {_cuda_lib} to LD_LIBRARY_PATH")

_has_cuda = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
TORCH_EXTRA = "gpu" if _has_cuda else "cpu"
print(f"CUDA detected: {_has_cuda} → using {TORCH_EXTRA}")
if not _has_cuda:
    print("WARNING: No GPU. Runtime → Change runtime type → GPU")

if DEP_MODE == "nanochat":
    # nanochat uses uv for dependency management
    NANOCHAT_DIR = os.path.join(REPO_DIR, "nanochat")
    if subprocess.run(["which", "uv"], capture_output=True).returncode != 0:
        run("curl -LsSf https://astral.sh/uv/install.sh | sh", shell=True)
        os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
    run(["uv", "sync", "--extra", TORCH_EXTRA], cwd=NANOCHAT_DIR)
    # Verify torch
    _check = subprocess.run(
        ["uv", "run", "python", "-c",
         "import torch; print(f'torch {torch.__version__}  cuda={torch.cuda.is_available()}')"],
        cwd=NANOCHAT_DIR, capture_output=True, text=True)
    print(_check.stdout.strip())
    if _has_cuda and "cuda=False" in _check.stdout:
        raise RuntimeError("torch installed but CUDA not available")

elif DEP_MODE == "pip":
    # Standalone experiment with pip dependencies
    pkgs = CFG.get("pip_packages", [])
    if pkgs:
        # Don't reinstall torch if Colab already has it
        skip = {"torch", "numpy"}
        to_install = [p for p in pkgs if p not in skip]
        if to_install:
            run([sys.executable, "-m", "pip", "install", "-q"] + to_install)
        print(f"Installed: {to_install} (skipped Colab builtins: {skip & set(pkgs)})")
    import torch
    print(f"torch {torch.__version__}  cuda={torch.cuda.is_available()}")

else:
    raise ValueError(f"Unknown dep_mode: {DEP_MODE}")

## 5. Restore artifacts from Drive

In [ ]:
import shutil, glob

os.makedirs(LOCAL_ARTIFACT_DIR, exist_ok=True)
os.makedirs(DRIVE_ARTIFACT_DIR, exist_ok=True)

drive_contents = list(Path(DRIVE_ARTIFACT_DIR).iterdir()) if Path(DRIVE_ARTIFACT_DIR).exists() else []
if drive_contents:
    print(f"Restoring artifacts from Drive ({len(drive_contents)} items)...")
    run(["rsync", "-a", "--timeout=60", "--info=progress2",
         f"{DRIVE_ARTIFACT_DIR}/", f"{LOCAL_ARTIFACT_DIR}/"])
else:
    print("No existing artifacts on Drive — fresh start.")

# For nanochat experiments, check for checkpoint to resume
resume_step = -1
if DEP_MODE == "nanochat":
    model_tag = CFG.get("model_tag", "d12")
    ckpt_dir = Path(LOCAL_ARTIFACT_DIR) / "base_checkpoints" / model_tag
    if ckpt_dir.exists():
        model_files = sorted(ckpt_dir.glob("model_*.pt"))
        if model_files:
            resume_step = int(model_files[-1].stem.split("_")[-1])
            print(f"Found checkpoint at step {resume_step} — will resume.")
    os.environ["NANOCHAT_BASE_DIR"] = LOCAL_ARTIFACT_DIR
    print(f"NANOCHAT_BASE_DIR={LOCAL_ARTIFACT_DIR}")
else:
    # For pip experiments, check for existing checkpoint
    ckpts = sorted(glob.glob(os.path.join(LOCAL_ARTIFACT_DIR, "checkpoints", "*.pth")))
    if ckpts:
        print(f"Found checkpoint: {Path(ckpts[-1]).name}")

## 6. Setup (if experiment requires it)

In [ ]:
setup_script = CFG.get("setup_script")

if setup_script:
    setup_path = os.path.join(REPO_DIR, setup_script)
    print(f"Running setup: {setup_script}")

    if DEP_MODE == "nanochat":
        # Check if tokenizer already exists (idempotent but slow)
        tokenizer_exists = bool(
            glob.glob(os.path.join(LOCAL_ARTIFACT_DIR, "tokenizer", "*.json"))
            or glob.glob(os.path.join(LOCAL_ARTIFACT_DIR, "tok*.model"))
            or glob.glob(os.path.join(LOCAL_ARTIFACT_DIR, "tokenizer*"))
        )
        n_shards = CFG.get("n_shards", 30)
        if tokenizer_exists:
            n_existing = len(glob.glob(os.path.join(LOCAL_ARTIFACT_DIR, "base_data_climbmix", "*.parquet")))
            print(f"Tokenizer found. Data shards: {n_existing}/{n_shards}")
            if n_existing >= n_shards:
                print("Skipping setup.")
                setup_script = None  # skip

    if setup_script:
        env = {
            **os.environ,
            "N_SHARDS": str(CFG.get("n_shards", 30)),
            "NANOCHAT_BASE_DIR": LOCAL_ARTIFACT_DIR,
            "TORCH_EXTRA": TORCH_EXTRA,
        }
        run(["bash", setup_path], env=env)
else:
    print("No setup script — skipping.")

## 7. Start background Drive sync

In [ ]:
import threading, time

_stop_sync = threading.Event()

def _sync_worker():
    while not _stop_sync.wait(timeout=SYNC_INTERVAL_MIN * 60):
        try:
            subprocess.run(
                ["rsync", "-a", "--timeout=60", f"{LOCAL_ARTIFACT_DIR}/", f"{DRIVE_ARTIFACT_DIR}/"],
                capture_output=True
            )
            print(f"[sync] {time.strftime('%H:%M:%S')} → Drive", flush=True)
        except Exception as e:
            print(f"[sync] WARNING: {e}", flush=True)

_sync_thread = threading.Thread(target=_sync_worker, daemon=True)
_sync_thread.start()
print(f"Background sync started (every {SYNC_INTERVAL_MIN} min → {DRIVE_ARTIFACT_DIR})")

## 8. Train

In [ ]:
train_script = CFG.get("train_script")
train_command = CFG.get("train_command")

env = {**os.environ}

if DEP_MODE == "nanochat":
    env.update({
        "NANOCHAT_BASE_DIR": LOCAL_ARTIFACT_DIR,
        "N_ITERATIONS": str(CFG.get("n_iterations", 6000)),
        "RESUME_FROM_STEP": str(resume_step),
    })
    if resume_step > 0:
        print(f"Resuming from step {resume_step}")
    else:
        print(f"Training from scratch for {CFG.get('n_iterations', 6000)} steps")

try:
    if train_script:
        run(["bash", os.path.join(REPO_DIR, train_script)], env=env, cwd=REPO_DIR)
    elif train_command:
        run(train_command, shell=True, env=env, cwd=REPO_DIR)
    else:
        raise ValueError("No train_script or train_command in colab.yaml")
finally:
    _stop_sync.set()
    print("\nFinal sync to Drive...")
    run(["rsync", "-a", "--timeout=60", "--info=progress2",
         f"{LOCAL_ARTIFACT_DIR}/", f"{DRIVE_ARTIFACT_DIR}/"])
    print("Done.")

## 9. Post-training (optional)

Runs the post_script if one is defined in colab.yaml (e.g. Phase 0 probes for VVVVVV).

In [ ]:
post_script = CFG.get("post_script")

if post_script:
    print(f"Running post-training script: {post_script}")
    env = {**os.environ}
    if DEP_MODE == "nanochat":
        env["NANOCHAT_BASE_DIR"] = LOCAL_ARTIFACT_DIR
    run(["bash", os.path.join(REPO_DIR, post_script)], env=env, cwd=REPO_DIR)

    # Sync main artifacts
    run(["rsync", "-a", f"{LOCAL_ARTIFACT_DIR}/", f"{DRIVE_ARTIFACT_DIR}/"])

    # Sync extra outputs if defined
    for extra in CFG.get("extra_sync", []):
        src = os.path.join(REPO_DIR, extra["src"])
        dst = os.path.join(DRIVE_ROOT, ARTIFACT_SUBDIR, extra["dst"])
        os.makedirs(dst, exist_ok=True)
        run(["rsync", "-a", f"{src}/", f"{dst}/"])
        print(f"Synced {extra['src']} → {dst}")
else:
    print("No post-training script defined.")